# FlowEdit → Modular Diffusers — smoke + E2E (training-free image editing)

Inversion-free text-based editing on FLUX (FlowEdit, arXiv:2412.08629). Publish PRIVATE
`remyxai/flowedit-flux-modular` → load via `trust_remote_code` → **smoke** (a tiny edit) → **E2E** (real edits +
a CLIP edit-direction metric). Upload `block.py` first. Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.

## 1 · Install + GPU + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16

## 2 · Publish PRIVATE (upload block.py first)

In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"FlowEditBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.FlowEditBlock"}},indent=2))
F="black-forest-labs/FLUX.1-dev"
def c(s,l,cl): return [None,None,{"pretrained_model_name_or_path":F,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"FlowEditBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c("text_encoder","transformers","CLIPTextModel"),"tokenizer":c("tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c("text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c("tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c("transformer","diffusers","FluxTransformer2DModel"),"vae":c("vae","diffusers","AutoencoderKL"),
 "scheduler":c("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/flowedit-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))

## 3 · Load + a source image

In [ ]:
from diffusers import ModularPipeline
from PIL import Image
from io import BytesIO
import requests
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/flowedit-flux-modular", trust_remote_code=True)
print("loaded block:", type(pipe.blocks).__name__)   # expect FlowEditBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

IMG_URL = "https://raw.githubusercontent.com/fallenshock/FlowEdit/main/inputs/cat.png"  #@param {type:"string"}
try:
    src = Image.open(BytesIO(requests.get(IMG_URL, timeout=30).content)).convert("RGB")
except Exception as e:
    print("fetch failed, upload one:", e)
    from google.colab import files; up=files.upload(); src=Image.open(list(up.keys())[0]).convert("RGB")
src = src.resize((1024,1024)); src.save("src.png")
print("source:"); display(src.resize((384,384)))

## 4 · Milestone A — smoke (tiny edit)

In [ ]:
import torch
g=torch.Generator(DEV).manual_seed(0)
sm = pipe(image="src.png", source_prompt="a cat", prompt="a dog",
          height=512, width=512, T_steps=12, n_max=10, generator=g).images[0]
print("[SMOKE] ran; output size", sm.size)
from IPython.display import display; display(sm)

## 5 · Milestone B — E2E edits (full settings)

In [ ]:
import torch
from PIL import Image
from IPython.display import display
EDITS = [("a cat","a dog"), ("a cat","a tiger")]   # (source_prompt, target_prompt)
outs=[("source","",src)]
for sp,tp in EDITS:
    g=torch.Generator(DEV).manual_seed(0)
    im=pipe(image="src.png", source_prompt=sp, prompt=tp, height=1024, width=1024,
            T_steps=28, src_guidance_scale=1.5, tar_guidance_scale=5.5, n_max=24, n_min=0, generator=g).images[0]
    im.save(f"edit_{tp.replace(' ','_')}.png"); outs.append((tp,"",im)); print("  ✓", tp)
S=512; W=len(outs)*S+(len(outs)+1)*10
row=Image.new("RGB",(W,S+40),"white")
from PIL import ImageDraw, ImageFont
d=ImageDraw.Draw(row)
try: F=ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",26)
except: F=ImageFont.load_default()
for i,(name,_,im) in enumerate(outs):
    x=10+i*(S+10); row.paste(im.resize((S,S)),(x,0)); d.text((x+8,S+6),name,fill="black",font=F)
row.save("flowedit_grid.png"); print("source | edits:"); display(row.resize((min(W,1400),int((S+40)*min(W,1400)/W))))

## 6 · Quantitative — CLIP edit-direction (did the edit move toward the target?)

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval()
proc=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
def sim(img, text):
    b=proc(text=[text], images=[img], return_tensors="pt", padding=True).to(DEV)
    with torch.no_grad(): o=clip(**b)
    ie=o.image_embeds/o.image_embeds.norm(dim=-1,keepdim=True); te=o.text_embeds/o.text_embeds.norm(dim=-1,keepdim=True)
    return float((ie@te.T)[0,0])
print("edit | CLIP(edit,target) vs CLIP(source,target) — edit should be higher")
ok=True
for sp,tp in EDITS:
    e=Image.open(f"edit_{tp.replace(' ','_')}.png")
    se, ss = sim(e,tp), sim(src,tp)
    print(f"  {tp:>10}:  edit={se:.3f}  source={ss:.3f}  -> {'PASS' if se>ss else 'REVIEW'}")
    ok = ok and se>ss
print("\nedit-direction:", "PASS" if ok else "REVIEW")

## Verdict
`loaded block: FlowEditBlock` + coherent edits that **preserve structure** while changing the target, with the
CLIP edit-direction PASS = the modular FlowEdit works. Optional stronger check: run `fallenshock/FlowEdit`'s
`FlowEditFLUX` on the same image/prompts and eyeball parity. Then flip public + collection.